## I2 reader mode: CONSTRUCTED rolling OOS allocation

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are separately labelled replay. French raw/extracted/processed bytes are not bundled. French route: A10-A limited-scope design accepted, execution blocked; A10-A.1 protocol technically accepted pending permissions and execution authorization. G01 rights/subject/use unknown; G05 A10-B not authorized.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 3：四种长仓组合的滚动样本外比较

本 Notebook 在内存中重跑冻结配置（projects/project03/configs/project03.json），调用 projects/project03/src/project03_backtest.py 与 src/quant_course 中的核心函数，不写任何文件。数据是 2026-10-05 冻结的 French 10 行业当前修订快照，非 PIT；结果是理想化月边界的描述性回测。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project03_portfolio_oos_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0,str(ROOT / "projects/project03/src"))
from project03_backtest import STRATEGIES, Spec, run_strategy, apply_costs, metrics
from quant_course.performance import wealth_index
idx = pd.period_range("2000-01",periods=48,freq="M")
rng = np.random.default_rng(103)
returns = pd.DataFrame(rng.normal(0.005,0.03,(48,4)),index=idx,columns=list("ABCD"))
rf = pd.Series(0.001,index=idx)
spec = Spec(window=24,max_weight=0.4)
print("CONSTRUCTED OOS method; French branch SKIPPED: rights unknown")


## 1. 运行全部规格（约 25 秒）

In [ ]:
ledgers = {}; weights = {}
for strategy in STRATEGIES:
    ledger, weights[strategy] = run_strategy(strategy,returns,rf,spec,"2002-01","2003-12")
    ledgers[strategy] = apply_costs(ledger,10.0)
primary = pd.DataFrame({s:metrics(l) for s,l in ledgers.items()}).T
display(primary)
primary.to_csv(RUN_DIR / "constructed_primary_metrics.csv")


## 2. 与已保存的 summary.json 一致（若已运行 CLI）

In [ ]:
print("Historical French summary comparison SKIPPED: this constructed panel cannot reproduce market results")


## 3. 账本恒等式与时点

In [ ]:
led = ledgers["max_sharpe"]
assert np.allclose(led["net_return"], (1-led["cost"])*(1+led["gross_return"])-1)
assert (pd.PeriodIndex(led["train_end"],freq="M") < pd.PeriodIndex(led["holding_month"],freq="M")).all()
display(led[["holding_month","train_start","train_end","turnover","gross_return","cost","net_return"]].head())
for s,l in ledgers.items(): l.to_csv(RUN_DIR / (s+"_ledger_CONSTRUCTED.csv"),index=False)


## 4. 净财富（对数刻度）

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for s,l in ledgers.items(): ax.plot(wealth_index(l["net_return"].to_numpy()),label=s)
ax.set(title="CONSTRUCTED rolling allocation",xlabel="Holding month",ylabel="Net wealth"); ax.legend()
fig.tight_layout(); fig.savefig(RUN_DIR / "project03_wealth_CONSTRUCTED.png",dpi=130); plt.show()


## 5. 相对等权的块自助法区间

In [ ]:
print("Full historical bootstrap/sensitivity grid SKIPPED; the small constructed path checks timing and cost accounting only")


## 6. 回退记录与后见之明对照

后见之明组合用整个评价期估计，不可交易，只用于说明样本内最优的高估。

In [ ]:
print(pd.concat([l.assign(strategy=s) for s,l in ledgers.items()]).groupby(["strategy","status"]).size())
